# 02 · Preparing knowledge for an application

**Session 01 · notebook 2 of 6**

## Objective

Turn a collection of documents into something an application can search: documents with metadata, split into chunks that keep their meaning, represented as vectors. See how the choices made here decide what retrieval can find later.

## The application after this notebook

```
documents ──> metadata ──> chunks ──> embeddings ──> index
                                                      │
question ──> [ instructions + retrieved context ] ──> model ──> answer     (notebook 03)
```

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/avsolatorio/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "pydantic", "scikit-learn", "requests"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook uses recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
llm = w.LLM()
print("Model client:", llm.mode())

Model client: recorded outputs (no WORKSHOP_API_KEY set)


## 1. The collection

Eight short documents. Each has a header with an id, a title, a type, a country, a date, and a source. Four are relevant briefs, one is a definition note, one is an outdated summary, and two are distractors. See `assets/README.md` for what each is for.

In [2]:
docs = w.load_documents()
print(f"{'id':20s} {'type':16s} {'date':12s} title")
for d in docs:
    print(f"{d.id:20s} {d.type:16s} {d.date:12s} {d.title}")

id                   type             date         title
coffee               sector note      2026-10-13   Coffee exports in East Africa
internet-indicator   indicator note   2026-10-13   How the internet-users indicator is defined and measured
kenya-digital        country brief    2026-10-13   Kenya: internet use and mobile subscriptions, 2010 to 2024
kenya-economy        country brief    2026-10-13   Kenya: GDP per capita and growth, 2010 to 2024
mobile-money         topic note       2026-10-13   Mobile money in Kenya and Rwanda
regional-2016        summary          2016-06-01   Regional digital summary, 2016
rwanda-digital       country brief    2026-10-13   Rwanda: internet use and mobile subscriptions, 2010 to 2024
rwanda-economy       country brief    2026-10-13   Rwanda: GDP per capita and growth, 2010 to 2024


## 2. Metadata is part of the knowledge

The outdated summary is the clearest case. Its text reads as current ("the latest available figures are for 2015") and its figures were correct when it was written. The only thing that tells an application to rank it below the 2026 briefs is its date, which lives in the header.

In [3]:
d = next(d for d in docs if d.id == "regional-2016")
print("date:  ", d.date)
print("type:  ", d.type)
print("note:  ", d.note)
print()
print(d.text[:500])

date:   2016-06-01
type:   summary
note:   Written for the training workshop as an example of an outdated document. Not an official publication.

# Regional digital summary, 2016

Internet use in East Africa continues to expand. In Kenya, 16.6 percent of the population used the internet in 2015, up from 7.2 percent in 2010. In Rwanda the share was 12.5 percent in 2015, up from 8.0 percent in 2010.

Mobile cellular subscriptions reached 80.1 per 100 people in Kenya and 75.3 per 100 people in Rwanda in 2015.

On current trends, internet use in both countries is expected to reach about 25 percent of the population by 2020 and about half of t


## 3. Chunking: fixed windows against sections

A chunk is the unit that retrieval returns and the model reads. Two strategies on the same brief:

- **Fixed windows** of 400 characters with an overlap. Simple, and blind to the document's structure.
- **Sections**, one chunk per Markdown heading, with the title and the heading kept in the text and the section name in the metadata.

In [4]:
kenya = next(d for d in docs if d.id == "kenya-digital")
fixed = w.chunk_fixed(kenya, size=400, overlap=50)
sections = w.chunk_sections(kenya)
print(f"fixed windows: {len(fixed)} chunks · sections: {len(sections)} chunks\n")

print("A fixed window (note where it starts and ends):\n")
print(repr(fixed[1].text))
print("\nA section chunk:\n")
print(sections[1].text)
print("\nmetadata:", sections[1].section, "|", sections[1].date, "|", sections[1].country)

fixed windows: 4 chunks · sections: 5 chunks

A fixed window (note where it starts and ends):

'able.\n\nOver the full period the share increased by about 28 percentage points. The fastest growth in absolute terms came between 2010 and 2015, when the share more than doubled.\n\n## Mobile subscriptions\n\nMobile cellular subscriptions per 100 people stood at 60.0 in 2010, 80.1 in 2015, and 117.6 in 2020. The figure was 121.2 in 2022, 120.6 in 2023, and 126.5 in 2024. Values above 100 reflect multip'

A section chunk:

Kenya: internet use and mobile subscriptions, 2010 to 2024
Internet use
The share of individuals using the internet in Kenya rose from 7.2 percent of the population in 2010 to 16.6 percent in 2015 and 24.7 percent in 2020. Growth continued after 2020: 29.4 percent in 2022, 32.1 percent in 2023, and 35.0 percent in 2024, the latest year available.

Over the full period the share increased by about 28 percentage points. The fastest growth in absolute terms came between 2010 

The fixed window starts and ends mid-sentence and carries no heading, so a passage about mobile subscriptions can be retrieved for a question about internet use without anything in the chunk saying which it is. The section chunk carries its title and heading, which both the retriever and the model can use.

## 4. Embeddings

An embedding is a vector that places a text so that texts with similar meaning are close. The `Embedder` uses a small sentence-transformer model when it can be downloaded and falls back to TF-IDF otherwise; both give the same interface. The cell reports which one is in use.

In [5]:
# Optional, for better embeddings in Colab (about 90 MB download):
# %pip install -q sentence-transformers

import numpy as np

chunks = [c for d in docs for c in w.chunk_sections(d)]
emb = w.Embedder()
emb.fit([c.text for c in chunks])
print("embedding backend:", emb.backend)

sample = [c for c in chunks if c.section in ("Internet use", "Mobile subscriptions", "Growth")][:6]
vectors = emb.encode([c.text for c in sample])
query = emb.encode(["share of people using the internet"])[0]
print()
for c, v in sorted(zip(sample, vectors), key=lambda cv: -float(cv[1] @ query)):
    print(f"{float(v @ query):6.3f}  {c.doc_id:16s} {c.section}")

embedding backend: tfidf

 0.177  rwanda-digital   Internet use
 0.171  kenya-digital    Internet use
 0.166  kenya-digital    Mobile subscriptions
 0.090  rwanda-digital   Mobile subscriptions
 0.000  kenya-economy    Growth
 0.000  rwanda-economy   Growth


Sections about internet use score highest for a query about internet use, the mobile-subscription sections next, and growth last. With the sentence-transformer backend the separation is wider; with TF-IDF it rests on shared words. The lesson holds for both: the quality of the ranking depends on what the chunk text contains.

## 5. What poor preparation does

The same collection with the headers stripped, every document concatenated into one text, and fixed windows. Search for the latest Kenya figure and compare the top results with the section index.

In [6]:
blob = w.Document(id="blob", title="", type="", country="", date="", source="", note="", text="\n\n".join(d.text for d in docs))
poor = w.Retriever(w.chunk_fixed(blob, size=400, overlap=50))
good = w.Retriever(chunks)

q = "What is the latest available figure for internet use in Kenya?"
print("Poorly prepared (one blob, fixed windows, no metadata):")
for h in poor.search(q, k=3):
    print(f"  {h.score:6.3f}  {h.chunk.text[:90]!r}")
print("\nPrepared (sections with metadata):")
w.print_hits(good.search(q, k=3))

Poorly prepared (one blob, fixed windows, no metadata):
   0.201  'h about 25 percent of the population by 2020 and about half of the population by 2025. Exp'
   0.140  '024\n\n## Internet use\n\nThe share of individuals using the internet in Rwanda was 8.0 percen'
   0.128  's from the share of individuals using the internet. The two indicators answer different qu'

Prepared (sections with metadata):
 0.221  [regional-2016 · intro · 2016]
 0.159  [kenya-digital · Internet use · 2026]
 0.135  [rwanda-digital · Internet use · 2026]


In the poorly prepared index the top results are fragments that cannot be cited and carry no date. In the prepared index each hit names its document, section, and year, so the application can filter and the reader can check.

## 6. Build the index for the next notebook

Notebook 03 rebuilds this index with one line. The retriever holds the chunks, the embedder, and the matrix of vectors.

In [7]:
retriever = w.Retriever(chunks)
print(len(retriever.chunks), "chunks indexed with", retriever.embedder.backend)

26 chunks indexed with tfidf


## Key points

- Metadata (date, type, country, section) is part of what the application knows. It decides what can be filtered and ranked, and what a citation can say.
- Chunk boundaries decide what retrieval can return. Sections keep meaning and headings; fixed windows cut through both.
- Embeddings place text by meaning; the quality of the index is the quality of the text and metadata it was built from.

## Exercise

Write a ninth document, a short brief on Uganda with figures looked up from the World Development Indicators, with the same header fields. Add it to `assets/documents/`, rerun this notebook, and check where its sections rank for the queries above.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.